<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px 30px 20px; border-radius: 10px; color: white; margin-bottom: 10px;">

# NexAU Cookbook — 构建数据库 Agent

输入 **"哪本书最贵？"**，从数据库中得到准确答案。

[![GitHub](https://img.shields.io/badge/NexAU-GitHub-blue?logo=github)](https://github.com/nex-agi/NexAU)

</div>

### 核心概念

<table style="width:100%; border-collapse: separate; border-spacing: 8px;">
<tr>
<td style="background:#e8f4fd; padding:12px; border-radius:8px; width:25%; vertical-align:top;">
<strong>LLM</strong><br><small>大语言模型</small><br>
理解你问题的 AI 模型<br><em>如 GPT-4、DeepSeek</em>
</td>
<td style="background:#e8f8e8; padding:12px; border-radius:8px; width:25%; vertical-align:top;">
<strong>Tool</strong><br><small>工具</small><br>
Agent 可调用的函数<br><em>如 <code>execute_sql</code></em>
</td>
<td style="background:#fff3e0; padding:12px; border-radius:8px; width:25%; vertical-align:top;">
<strong>Skill</strong><br><small>技能</small><br>
教 Agent 理解数据的知识文件<br><em>SKILL.md</em>
</td>
<td style="background:#f3e8fd; padding:12px; border-radius:8px; width:25%; vertical-align:top;">
<strong>Agent</strong><br><small>智能体</small><br>
自主行动的 LLM<br><em>调用工具 → 解读 → 回答</em>
</td>
</tr>
</table>

### 教程路线图

| 章节 | 内容 | 你会得到 |
|:---:|:---|:---|
| **1-2** | 环境准备 & 创建示例数据库 | 一个有坑的书店数据库 |
| **3** | 构建 `execute_sql` 工具 | 三层安全的 SQL 执行器 |
| **4** | 运行基础 Agent | 能用，但会犯错 |
| **5-7** | Skill + 自动生成 + System Prompt | 领域知识注入 |
| **8** | 运行完整 Agent | 准确回答，对比提升 |

---

### 1. 环境准备

#### 1.1 安装依赖

安装 [NexAU](https://github.com/nex-agi/NexAU)：

In [ ]:
# 从 GitHub Release 安装 NexAU
%pip install https://github.com/nex-agi/NexAU/releases/download/v0.4.1/nexau-0.4.1-py3-none-any.whl

#### 1.2 导入模块

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import time
from pathlib import Path
from typing import Any

---

### 2. 创建示例数据库

我们用一个小型**书店**场景，包含 3 张表：

<div style="background:#f8f9fa; padding:15px; border-radius:8px; font-family:monospace; font-size:14px;">
customers（客户） ◄──┐<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;│ customer_id<br>
orders（订单）&nbsp;&nbsp;&nbsp;────┤<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;│ book_id<br>
books（图书）&nbsp;&nbsp;&nbsp;&nbsp;◄──┘
</div>

<div class="alert alert-warning" style="margin-top:15px;">
<strong>⚠️ 注意：</strong>这个数据库故意埋了几个<strong>坑</strong> — 和真实数据库一样：
<ul>
<li><code>price</code> 是 TEXT 类型 — <code>ORDER BY price</code> 按字母序排列，<code>"89" > "168"</code></li>
<li><code>status</code> 包含 "已取消" — 不过滤的话收入会算错</li>
<li><code>member_level</code> 是中文文本 — 没有数字等级</li>
</ul>
</div>

In [ ]:
DB_PATH = Path("sample_bookstore.sqlite")

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# -- customers -------------------------------------------------------
cur.execute(
    "CREATE TABLE customers ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  name        TEXT    NOT NULL,"
    "  email       TEXT    UNIQUE,"
    "  city        TEXT,"
    "  member_level TEXT   DEFAULT '普通',"
    "  created_at  TEXT    DEFAULT (datetime('now'))"
    ")"
)
customers = [
    ("张三", "zhangsan@example.com", "北京", "金卡"),
    ("李四", "lisi@example.com", "上海", "普通"),
    ("王五", "wangwu@example.com", "广州", "银卡"),
    ("赵六", "zhaoliu@example.com", "深圳", "钻石"),
    ("孙七", "sunqi@example.com", "杭州", "普通"),
    ("周八", "zhouba@example.com", "成都", "金卡"),
    ("吴九", "wujiu@example.com", "北京", "银卡"),
    ("郑十", "zhengshi@example.com", "上海", "普通"),
    ("钱十一", "qian11@example.com", "广州", "金卡"),
    ("陈十二", "chen12@example.com", "深圳", "普通"),
]
cur.executemany(
    "INSERT INTO customers (name, email, city, member_level) VALUES (?, ?, ?, ?)",
    customers,
)

# -- books -----------------------------------------------------------
cur.execute(
    "CREATE TABLE books ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  title       TEXT    NOT NULL,"
    "  author      TEXT    NOT NULL,"
    "  genre       TEXT,"
    "  price       TEXT    NOT NULL,"
    "  stock       INTEGER DEFAULT 0,"
    "  publisher   TEXT,"
    "  publish_year INTEGER"
    ")"
)
books = [
    ("三体", "刘慈欣", "科幻", "36.00", 150, "重庆出版社", 2008),
    ("活着", "余华", "文学", "29.00", 200, "作家出版社", 1993),
    ("Python编程从入门到实践", "Eric Matthes", "技术", "89.00", 80, "人民邮电出版社", 2020),
    ("明朝那些事儿", "当年明月", "历史", "168.00", 60, "浙江人民出版社", 2009),
    ("原则", "瑞·达利欧", "经管", "98.00", 45, "中信出版社", 2018),
    ("流浪地球", "刘慈欣", "科幻", "32.00", 120, "中国华侨出版社", 2000),
    ("深度学习", "Ian Goodfellow", "技术", "168.00", 30, "人民邮电出版社", 2017),
    ("百年孤独", "马尔克斯", "文学", "55.00", 90, "南海出版公司", 2011),
    ("人类简史", "尤瓦尔·赫拉利", "历史", "68.00", 75, "中信出版社", 2014),
    ("从零到一", "彼得·蒂尔", "经管", "45.00", 55, "中信出版社", 2015),
    ("设计模式", "GoF", "技术", "79.00", 40, "机械工业出版社", 2000),
    ("围城", "钱钟书", "文学", "25.00", 180, "人民文学出版社", 1947),
]
cur.executemany(
    "INSERT INTO books (title, author, genre, price, stock, publisher, publish_year) "
    "VALUES (?, ?, ?, ?, ?, ?, ?)",
    books,
)

# -- orders ----------------------------------------------------------
cur.execute(
    "CREATE TABLE orders ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  customer_id INTEGER NOT NULL REFERENCES customers(id),"
    "  book_id     INTEGER NOT NULL REFERENCES books(id),"
    "  quantity    INTEGER NOT NULL DEFAULT 1,"
    "  total_price TEXT    NOT NULL,"
    "  order_date  TEXT    NOT NULL,"
    "  status      TEXT    DEFAULT '已完成'"
    ")"
)
orders = [
    (1, 1, 2, "72.00", "2025-01-15", "已完成"),
    (1, 3, 1, "89.00", "2025-02-20", "已完成"),
    (2, 2, 1, "29.00", "2025-01-10", "已完成"),
    (2, 5, 1, "98.00", "2025-03-01", "待发货"),
    (3, 1, 1, "36.00", "2025-02-14", "已完成"),
    (3, 8, 2, "110.00", "2025-03-05", "已完成"),
    (4, 7, 1, "168.00", "2025-01-22", "已完成"),
    (4, 4, 1, "168.00", "2025-02-28", "已取消"),
    (5, 6, 3, "96.00", "2025-03-10", "待发货"),
    (6, 3, 1, "89.00", "2025-01-05", "已完成"),
    (6, 11, 1, "79.00", "2025-02-15", "已完成"),
    (7, 9, 1, "68.00", "2025-03-12", "已完成"),
    (8, 12, 1, "25.00", "2025-01-20", "已完成"),
    (9, 10, 2, "90.00", "2025-02-25", "待发货"),
    (10, 1, 1, "36.00", "2025-03-15", "已完成"),
]
cur.executemany(
    "INSERT INTO orders (customer_id, book_id, quantity, total_price, order_date, status) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    orders,
)

conn.commit()
conn.close()

print(f"Database created: {DB_PATH}")
print(f"  Tables: customers ({len(customers)} rows), books ({len(books)} rows), orders ({len(orders)} rows)")

看一下每张表的前几行数据：

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

for table in ["customers", "books", "orders"]:
    rows = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchall()
    print(f"\n-- {table}（前 3 行） --")
    for r in rows:
        print(dict(r))

conn.close()

---

### 3. 构建 `execute_sql` 工具

这是 Agent 与数据库对话的**唯一通道**。

<div style="background:#e8f4fd; border-left:4px solid #1976d2; padding:12px 15px; border-radius:0 8px 8px 0; margin:10px 0;">
💡 <strong>打个比方：</strong><code>execute_sql</code> 就像银行的防弹玻璃窗口 — 客户（LLM）可以问问题、拿答案，但绝对伸不进去动金库。
</div>

**三层安全机制：**

<table style="width:100%; border-collapse: separate; border-spacing: 0 6px;">
<tr><td style="background:#c8e6c9; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 1 层：关键字检查</strong> — SQL 是 SELECT 开头吗？
</td></tr>
<tr><td style="background:#fff9c4; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 2 层：注释剥离</strong> — 有没有人把 DELETE 藏在注释后面？
</td></tr>
<tr><td style="background:#ffcdd2; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 3 层：只读连接</strong> — 即使前两层漏了，SQLite 也拒绝写入
</td></tr>
</table>

**结构化输出：** 返回 JSON 字典（`status`、`data`、`warnings`），查询返回 0 行时自动提示模型检查条件 — 这就是 Agent **自我纠错**的机制。

#### 3.1 实现代码

In [ ]:
MAX_ROWS = 10
MAX_OUTPUT_LENGTH = 50_000
DEFAULT_TIMEOUT = 30

_DANGEROUS_KEYWORDS = (
    "DROP", "TRUNCATE", "DELETE", "ALTER", "CREATE",
    "INSERT", "UPDATE", "REPLACE", "ATTACH", "DETACH",
    "PRAGMA", "VACUUM", "REINDEX", "GRANT", "REVOKE",
)


def _strip_sql_comments(sql: str) -> str:
    """剥除 -- 和 /* */ 注释，防止注入绕过。"""
    no_line = re.sub(r"--[^\n]*", "", sql)
    no_block = re.sub(r"/\*.*?\*/", "", no_line, flags=re.DOTALL)
    return no_block


def execute_sql(
    sql: str,
    db_path: str | Path = DB_PATH,
    timeout: int = DEFAULT_TIMEOUT,
    max_rows: int = MAX_ROWS,
) -> dict[str, Any]:
    """
    执行只读 SELECT 查询，返回结构化结果。

    返回值字段：
      - status: "success" | "error" | "timeout"
      - columns, data, total_rows, row_count, truncated
      - warnings: 给模型的提示列表
    """
    start = time.time()

    # -- 第 1 层：关键字检查 --
    if not sql or not sql.strip():
        return {"status": "error", "error": "SQL query cannot be empty",
                "duration_ms": int((time.time() - start) * 1000)}

    cleaned_upper = _strip_sql_comments(sql).strip().upper()
    for kw in _DANGEROUS_KEYWORDS:
        if re.match(rf"^{kw}(?:\s|$)", cleaned_upper):
            return {"status": "error",
                    "error": f"Only SELECT queries allowed. Found: {kw}",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}

    if not re.match(r"^(SELECT|WITH)\b", cleaned_upper):
        return {"status": "error",
                "error": "Only SELECT or WITH ... SELECT queries allowed.",
                "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}

    # -- 第 3 层：只读连接 --
    connection = None
    try:
        uri = f"file:{db_path}?mode=ro"
        connection = sqlite3.connect(uri, uri=True)
        connection.row_factory = sqlite3.Row

        deadline = time.time() + timeout
        connection.set_progress_handler(
            lambda: 1 if time.time() > deadline else 0, 1000
        )

        cursor = connection.execute(sql)
        col_names = [d[0] for d in cursor.description] if cursor.description else []

        # 用 fetchmany 而不是 fetchall，避免大表无 LIMIT 时内存爆炸
        batch = cursor.fetchmany(max_rows + 1)
        truncated = len(batch) > max_rows
        rows_to_convert = batch[:max_rows]
        total_rows = max_rows + 1 if truncated else len(batch)

        data = [dict(r) for r in rows_to_convert]
        duration_ms = int((time.time() - start) * 1000)

        warnings: list[str] = []
        if total_rows == 0:
            warnings.append(
                "Query returned 0 rows. Check: table name, column names, "
                "WHERE conditions, data availability."
            )
        if truncated:
            warnings.append(
                f"Showing {max_rows} rows (more available). "
                f"Add WHERE clauses or LIMIT to narrow results."
            )

        result: dict[str, Any] = {
            "status": "success",
            "sql": sql,
            "columns": col_names,
            "data": data,
            "row_count": len(data),
            "total_rows": total_rows,
            "truncated": truncated,
            "duration_ms": duration_ms,
        }

        # 防止单行数据过大撑爆输出
        if len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH:
            while (
                len(data) > 1
                and len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH
            ):
                data = data[:-1]
                result["data"] = data
                result["row_count"] = len(data)
                result["truncated"] = True
            if not any("length limit" in w for w in warnings):
                warnings.append(
                    "Results truncated due to output length limit. "
                    "Select fewer columns or add WHERE clauses."
                )

        if warnings:
            result["warnings"] = warnings
        return result

    except sqlite3.OperationalError as e:
        msg = str(e)
        if "interrupted" in msg.lower():
            return {"status": "timeout",
                    "error": f"Query timed out after {timeout}s",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}
        return {"status": "error", "error": msg, "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    except Exception as e:
        return {"status": "error", "error": str(e), "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    finally:
        if connection:
            connection.close()

#### 3.2 测试工具

验证一下正常查询和安全拦截：

In [ ]:
# 正常查询
result = execute_sql("SELECT title, price FROM books LIMIT 3")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 空结果会触发 warning，提示模型检查条件
result = execute_sql("SELECT * FROM books WHERE genre = '玄幻'")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 安全测试：拦截 DROP — 第 1 层拦住
result = execute_sql("DROP TABLE books")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 安全测试：注释绕过 — 第 2 层剥除注释后，第 1 层拦住 DELETE
result = execute_sql("-- just a comment\nDELETE FROM books")
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 3.3 TEXT 当数字的陷阱

<div class="alert alert-danger">
<strong>🐛 头号陷阱：</strong>当 <code>price</code> 以 TEXT 存储时，<code>ORDER BY price</code> 按<strong>字母序</strong>排列 — <code>"89.00" > "168.00"</code>，因为 <code>"8" > "1"</code>。
<br><br>
解决方法：<code>CAST(price AS REAL)</code> — 排序前把 TEXT 转成数字。但模型不会自动知道这一点 — 这就是 Skill 的用武之地（第 5 节）。
</div>

In [ ]:
# 错误：TEXT 排序 — "89.00" 排在 "168.00" 前面（字母序）
result = execute_sql("SELECT title, price FROM books ORDER BY price DESC LIMIT 5")
print("TEXT 排序（错误）：")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

print()

# 正确：CAST 转为数字后排序
result = execute_sql("SELECT title, CAST(price AS REAL) AS price FROM books ORDER BY price DESC LIMIT 5")
print("数值排序（正确）：")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

---

### 4. 初次体验 — 运行基础 Agent

有了 `execute_sql`，就够搭一个能用的 Agent 了。现在就动手！

<div style="background:#fff3e0; border-left:4px solid #ff9800; padding:12px 15px; border-radius:0 8px 8px 0;">
🎯 这里故意只给一句话的提示词。等你看到基础 Agent 在哪里翻车，第 5-7 节的必要性就不言自明了。
</div>

#### 4.1 配置 LLM

填入你的 API 信息。NexAU 使用 **OpenAI 兼容接口**格式 — 任何遵循相同协议的 LLM 提供商都可以直接使用（OpenAI、DeepSeek、Moonshot、Ollama 本地模型等）。

In [ ]:
import os

os.environ["LLM_API_KEY"]  = "sk-xxx"                        # 你的 API Key
os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"     # 或：https://api.deepseek.com
os.environ["LLM_MODEL"]    = "gpt-4o-mini"                   # 或：deepseek-chat

#### 4.2 创建一个最简 Agent

把 `execute_sql` 包装成 NexAU 的 `Tool`，给 Agent 一句话指令，就能跑了：

In [ ]:
from nexau import Agent, AgentConfig, LLMConfig, Tool

sql_tool = Tool(
    name="execute_sql",
    description=(
        "Execute a SQL query against the SQLite database. "
        "Only SELECT queries are allowed."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "sql": {
                "type": "string",
                "description": "The SQL query to execute"
            },
            "max_rows": {
                "type": "integer",
                "description": "Maximum rows to return (default 10)",
                "default": 10
            }
        },
        "required": ["sql"]
    },
    implementation=execute_sql,
)

basic_agent = await Agent.create(config=AgentConfig(
    name="basic_bookstore_agent",
    system_prompt="你是一个数据库助手。使用 execute_sql 工具回答关于书店数据库的问题。",
    llm_config=LLMConfig(),
    tools=[sql_tool],
))

print("基础 Agent 已就绪！")

#### 4.3 试一试

In [ ]:
response = await basic_agent.run_async(message="数据库里有哪些表？")
print(response)

In [ ]:
response = await basic_agent.run_async(message="哪本书最贵？")
print(response)

#### 4.4 基础 Agent 的问题

基础 Agent 能用 — 但跑下面这些问题时，仔细观察：

<div class="alert alert-warning">
<strong>观察要点：</strong>
<ul>
<li>价格排序用了 <code>CAST(price AS REAL)</code> 吗？（大概率没有）</li>
<li>计算收入时排除了已取消订单吗？（它不知道应该排除）</li>
<li>它知道 <code>member_level</code> 有哪些取值吗？（只能猜）</li>
</ul>
多试几个你会发现：<strong>模型不了解你数据的特殊之处。</strong>接下来几节就是解决这个问题的。
</div>

In [ ]:
# 很可能得到错误结果 — TEXT 排序而不是数值排序
response = await basic_agent.run_async(message="列出价格最高的5本书")
print(response)

In [ ]:
# 它知道要排除已取消的订单吗？
response = await basic_agent.run_async(message="2025年3月的总收入是多少？")
print(response)

---

### 5. 编写 SKILL.md — 教模型认识你的数据

模型有了工具，但还不知道*怎么*正确查询。

<div style="background:#e8f4fd; border-left:4px solid #1976d2; padding:12px 15px; border-radius:0 8px 8px 0; margin:10px 0;">
💡 <strong>打个比方：</strong>招了新数据分析师，你不会只给数据库权限就说"去吧"。你会给他数据字典、指出陷阱、示范查询。<strong>SKILL.md 就是这个作用。</strong>
</div>

Skill 是 [NexAU](https://github.com/nex-agi/NexAU) 注入领域知识的标准机制。

#### 5.1 SKILL.md 的结构

| 章节 | 作用 | 类比 |
|:---|:---|:---|
| `description` | 模型判断"我需要这个 Skill 吗？" | 文件柜抽屉的标签 |
| **When to use** | 哪些问题应触发这个 Skill | "问这类问题时打开这个抽屉" |
| **Schema** | 列名、类型、主键、外键 | 数据字典 |
| **Common values** | 枚举值 | 贴在桌上的速查表 |
| **Example queries** | 2-3 个正确 SQL 示例 | "上一个分析师是这么查的" |
| **Gotchas** | 陷阱和注意事项 | **最有价值** |

<div class="alert alert-info">
<strong>💎 Gotchas 是最有价值的章节。</strong>模型不会因为找不到表而出错 — 它出错是因为不知道数据里的坑。
</div>

#### 5.2 完整示例：`books` 表的 SKILL.md

In [ ]:
books_skill = '''---
name: books
description: >-
  Use this skill whenever the user asks about books — titles, authors, genres,
  prices, stock levels, or publishers. Join to orders via book_id.
---

# books

The complete book catalog. One row per book, keyed by `id`.

## When to use

- "What science fiction books do we have?"
- "Which book is the most expensive?"
- "How many books by 刘慈欣?"

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Book ID — join key for `orders.book_id` |
| `title` | TEXT | Book title |
| `author` | TEXT | Author name |
| `genre` | TEXT | Category: `文学` / `技术` / `历史` / `科幻` / `经管` |
| `price` | TEXT | Price in yuan — **TEXT not numeric**, use `CAST(price AS REAL)` |
| `stock` | INTEGER | Inventory count |
| `publisher` | TEXT | Publisher name |
| `publish_year` | INTEGER | Year of publication |

## Common values

- `genre`: `文学`, `技术`, `历史`, `科幻`, `经管`

## Example queries

**Most expensive books:**

```sql
SELECT title, author, CAST(price AS REAL) AS price_yuan
FROM books
ORDER BY price_yuan DESC
LIMIT 5;
```

## Gotchas

- `price` is **TEXT**, not REAL — always `CAST(price AS REAL)` for numeric ops.
- `genre` uses Chinese category names. For fuzzy search use `LIKE '%技术%'`.
'''

print(books_skill)

#### 5.3 反模式

| 不要这样做 | 会出什么问题 |
|---|---|
| `description: "关于图书的信息"` | 太笼统 — 模型无法判断何时该读 |
| 省略 Gotchas | 模型反复踩 TEXT 当数字的坑 |
| 示例查询用 `SELECT *` | 模型照搬，结果集爆炸 |
| 把所有知识塞进 System Prompt | 每轮都浪费 token，模型注意力被稀释 |
| 一个 Skill 管 5 张表 | 无法按需加载 — 要么全读，要么不读 |

---

### 6. 自动生成 Skill

3 张表手写没问题，30 张表就需要自动化了。下面的脚本读取任意 SQLite，为每张表生成 SKILL.md 草稿：

- 表结构（列名、类型、主键、外键）
- TEXT 列实际存储数字的情况
- 枚举型列（取值较少的列）
- 外键关系（自动生成 JOIN 示例）

需要人工判断的地方标记了 `[TODO]`。

#### 6.1 辅助函数

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    """Return all user table names."""
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    """Return column metadata for a table."""
    # Table names come from sqlite_master, so they are trusted identifiers.
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    """Return FK mapping {local_col: 'ref_table.ref_col'}."""
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    """Return the most common values in a column."""
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    """Detect if a TEXT column actually stores numeric values."""
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False

#### 6.2 主生成函数

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    """Generate a SKILL.md draft for a single table."""
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "(no explicit PK)"

    fk_notes = [f"`{lc}` -> `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK -> `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " | ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            val_str = ", ".join(f"`{v[0]}`" for v in vals)
            common_sections.append(f"- `{col['name']}`: {val_str}")

    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** | PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

#### 6.3 在示例数据库上运行

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"Found {len(tables)} tables: {', '.join(tables)}\n")

for table in tables:
    print("=" * 60)
    print(f"  {table}")
    print("=" * 60)
    print(generate_skill_md(conn, table))

conn.close()

看看自动生成器发现了什么：

- `books.price` 和 `orders.total_price` 被标记为"TEXT 存储数字" — 头号陷阱
- 外键关系被检测到，自动生成了 JOIN 示例
- 枚举列（`status`、`genre`、`member_level`）的取值被列出
- `[TODO]` 标记了需要你补充业务上下文的地方

---

### 7. System Prompt（系统提示词）

**系统提示词**是 Agent 的操作手册，定义了它的工作方式：

```
规划 → 写 SQL → 执行 → 反思 → 回答
```

它还把 Skill 中的关键知识（CAST 注意事项、排除已取消订单、枚举值）嵌入提示词，让 Agent 每轮都能看到。

> **为什么不只依赖 Skill？** 完整 NexAU 部署中 Skill 是动态加载的，本教程为简单起见直接写进系统提示词。

In [ ]:
SYSTEM_PROMPT = """
You are a database agent. Your job: translate natural-language questions
into correct SQL, execute the query, and return a clear answer grounded in the data.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: `customers`, `books`, `orders`
- Primary join keys: `orders.customer_id` -> `customers.id`, `orders.book_id` -> `books.id`

## Table Knowledge (Skills)

### books
- Columns: id (PK), title, author, genre, price (TEXT!), stock, publisher, publish_year
- genre values: 文学, 技术, 历史, 科幻, 经管
- GOTCHA: `price` is TEXT — use CAST(price AS REAL) for numeric operations

### customers
- Columns: id (PK), name, email, city, member_level, created_at
- member_level values: 普通, 银卡, 金卡, 钻石 (no numeric rank)

### orders
- Columns: id (PK), customer_id (FK->customers), book_id (FK->books), quantity, total_price (TEXT!), order_date, status
- status values: 已完成, 待发货, 已取消
- GOTCHA: `total_price` is TEXT — use CAST(total_price AS REAL)
- GOTCHA: Exclude `status = '已取消'` from revenue calculations
- GOTCHA: `total_price` is pre-computed; don't re-multiply quantity * price

## Workflow

1. **Plan.** Identify which tables you need.
2. **Write the SQL.** SQLite syntax. Always use LIMIT. Prefer explicit columns over SELECT *.
3. **Execute.** Call `execute_sql`.
4. **Reflect.** If 0 rows or warnings, reconsider your query — maybe wrong column name or too-strict filter.
5. **Answer** in the user's language. Be concise. Show the key data.

## Constraints

- Only SELECT queries work — the tool rejects everything else.
- Don't hallucinate columns. If unsure, query the schema first.
"""

print(SYSTEM_PROMPT)

---

### 8. 完整版 Agent

<div class="alert alert-success">
<strong>✅ 升级！</strong>用完整的系统提示词 + 领域知识重新构建 Agent，和第 4 节的基础版对比效果。
</div>

#### 8.1 创建完整版 Agent

同样的 `sql_tool`，但这次配上了完整的系统提示词：

In [ ]:
from nexau import Agent, AgentConfig, LLMConfig, Tool

sql_tool = Tool(
    name="execute_sql",
    description=(
        "Execute a read-only SQL query against the bookstore SQLite database. "
        "Only SELECT queries are allowed. Results are capped at max_rows. "
        "Always use LIMIT. Use CAST for TEXT columns that store numbers."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "sql": {
                "type": "string",
                "description": "The SQL query to execute (SELECT only)."
            },
            "max_rows": {
                "type": "integer",
                "description": "Maximum rows to return. Default 10.",
                "default": 10
            }
        },
        "required": ["sql"]
    },
    implementation=execute_sql,
)

agent = await Agent.create(config=AgentConfig(
    name="bookstore_agent",
    system_prompt=SYSTEM_PROMPT,
    llm_config=LLMConfig(),
    tools=[sql_tool],
))

print(f"完整版 Agent 已就绪：{agent.config.name}")

#### 8.2 与基础版对比

用**和第 4 节完全相同的问题**来对比：

<div style="background:#e8f5e9; border-left:4px solid #4caf50; padding:12px 15px; border-radius:0 8px 8px 0;">
🔬 完整版 Agent 拥有 Skill 中的知识：TEXT 转数字、排除已取消订单、枚举值等。看看它是否犯同样的错。
</div>

In [ ]:
# 第 4 节得到了错误结果 — TEXT 排序把 98.00 排在 168.00 前面
# 完整版会用 CAST(price AS REAL) 吗？
response = await agent.run_async(message="列出价格最高的5本书")
print(response)

In [ ]:
# 第 4 节把已取消的订单也算进去了
# 完整版会排除 status = '已取消' 吗？
response = await agent.run_async(message="2025年3月的总收入是多少？")
print(response)

In [ ]:
# 它知道 member_level 的取值吗？
response = await agent.run_async(message="钻石会员有几个？分别是谁？")
print(response)

#### 8.3 试试你自己的问题！

改一下下面的问题，比如：
- "哪个城市的客户最多？"
- "刘慈欣的书总共卖了多少钱？"
- "库存最少的3本书是什么？"
- "每个月的订单量分别是多少？"

In [ ]:
# 改成你想问的任何问题！
response = await agent.run_async(message="每种类型的书平均价格是多少？")
print(response)

---

### 9. 总结

你分两步构建了一个数据库 Agent：

<table style="width:100%; border-collapse: separate; border-spacing: 0 6px;">
<tr>
<td style="background:#ffcdd2; padding:12px 15px; border-radius:8px; width:50%;">
<strong>基础版</strong>（第 4 节）<br>
<code>execute_sql</code> + 一句话提示词<br>
<em>能用，但会犯错</em>
</td>
<td style="background:#c8e6c9; padding:12px 15px; border-radius:8px; width:50%;">
<strong>完整版</strong>（第 8 节）<br>
+ SKILL.md + 完整系统提示词<br>
<em>回答准确，了解领域知识</em>
</td>
</tr>
</table>

NexAU 核心构建模块：

| 概念 | 作用 |
|:---|:---|
| **Tool** | Agent 可调用的函数（`execute_sql`） |
| **Skill** | 注入上下文的领域知识（SKILL.md） |
| **Agent** | 编排循环（`agent.run()`） |

---

<div class="alert alert-info">
<strong>🚀 用在你自己的数据库上：</strong>
<ol>
<li>把 <code>DB_PATH</code> 换成你的 SQLite 文件</li>
<li>运行 <code>generate_skills</code> 自动生成 SKILL.md 草稿</li>
<li>补充 <code>[TODO]</code> 处的业务知识</li>
<li>更新 <code>SYSTEM_PROMPT</code></li>
<li><code>agent.run(message="你的问题")</code> — 搞定！</li>
</ol>
生产环境部署请参考 <a href="https://github.com/nex-agi/NexAU">NexAU 文档</a>。
</div>

In [ ]:
# 清理
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"已清理 {DB_PATH}")